# Assignment 2: Milestone I Natural Language Processing
## Task 2&3
#### Student Name: XXXX XXXX
#### Student ID: 000000


Environment: Python 3 and Jupyter notebook

Libraries used: please include all the libraries you used in your assignment, e.g.,:
* pandas
* re
* numpy

## Introduction
You should give a brief information of this assessment task here.

<span style="color: red"> Note that this is a sample notebook only. You will need to fill in the proper markdown and code blocks. You might also want to make necessary changes to the structure to meet your own needs. Note also that any generic comments written in this notebook are to be removed and replace with your own words.</span>

## Install fasttest used for vector embedding

In [ ]:
!pip install --upgrade pip setuptools wheel
!pip install fasttext-wheel


## Importing libraries 

In [1]:
# Code to import libraries as you need in this assessment, e.g.,
import pandas as pd
import ast
from collections import Counter
import fasttext
import ast
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

## Task 2. Generating Feature Representations for Clothing Items Reviews

...... Sections and code blocks on buidling different document feature represetations


<span style="color: red"> You might have complex notebook structure in this section, please feel free to create your own notebook structure. </span>

### Load neccessary csv 
processed.csv

In [2]:
# Code to perform the task...
# === Task 2: Bag-of-Words ===

# Load processed.csv (must contain column 'tokens' from Task 1)
df = pd.read_csv("processed.csv")

# Convert 'tokens' from string to list if needed
print("[OK] processed.csv loaded. Rows:", len(df))

[OK] processed.csv loaded. Rows: 19662


### Building the Vocabulary Dictionary

reading the vocabulary.txt file and building a dictionary that maps each word to its index.

In [3]:
# === Task 2: Bag-of-Words ===
word2idx = {}
with open("vocabulary.txt", "r", encoding="utf-8") as f:
    for line in f:
        s = line.strip()
        if not s:
            continue
        w, sidx = s.rsplit(":", 1)
        word2idx[w] = int(sidx)

print("vocabulary.txt loaded. Size:", len(word2idx))
print("Sample entries:", list(word2idx.items())[:5])

vocabulary.txt loaded. Size: 7529
Sample entries: [('a-cup', 0), ('a-flutter', 1), ('a-frame', 2), ('a-kind', 3), ('a-line', 4)]


### Saving outputs
Save the count vector representation as per spectification.
- count_vectors.txt

In [4]:
# code to save output data...
# === Task 2: Bag-of-Words ===
from collections import Counter

out_path = "count_vectors.txt"

with open(out_path, "w", encoding="utf-8") as fout:
    for review_index, tokens in enumerate(df["tokens"].values):
        ctr = Counter()
        tokens = ast.literal_eval(tokens)
        for t in tokens:
            idx = word2idx.get(t)
            if idx is not None:
                ctr[idx] += 1

        # Format: "#<review_index>,idx:count,idx:count,..."
        
        parts = [f"{i}:{ctr[i]}" for i in sorted(ctr)]
        fout.write(f"#{review_index},{','.join(parts)}\n")

print(f"[OK] Wrote {len(df)} lines to {out_path}")
with open(out_path, "r", encoding="utf-8") as f:
    for _ in range(3):
        print(f.readline().rstrip())

[OK] Wrote 19662 lines to count_vectors.txt
#0,686:1,1027:1,1715:1,1791:1,2288:1,2481:1,2602:1,2892:2,3010:1,3087:1,3193:1,3258:1,3549:2,3552:1,3832:1,3934:1,4224:2,4234:1,4427:1,4639:2,5260:1,5668:1,6726:1,7092:1,7207:1,7406:1,7520:1,7522:1
#1,1286:1,2283:1,2502:1,2667:1,3403:1,6739:1
#2,86:1,924:1,1987:1,2646:1,3584:1,3595:1,4506:1,5736:2,5924:1,6716:1


### Extract the feature vector for each review
#### Unweigted Fasttext model

##### Print out dataframe tokens column's type

In [5]:
print(type(df['tokens'].iloc[0]))

<class 'str'>


##### Create a corpus file  
The corpus file represents distinct words of tokens, which will be used in the following steps.

In [6]:
with open("corpus.txt", "w", encoding="utf-8") as f:
    for tokens_str in df['tokens']:
        tokens_list = ast.literal_eval(tokens_str)
        f.write(" ".join(tokens_list) + "\n")

Now we train the skipgram fasttext model based on the corpus file

In [7]:
model = fasttext.train_unsupervised("corpus.txt", model="skipgram")

Try to print out sentence vector of one line to see the format

In [8]:
sentence = df['tokens'].iloc[0]
print(model.get_sentence_vector(sentence))

[-0.05077787  0.11196636 -0.08607846 -0.04829326  0.02529795  0.00220308
 -0.02636287  0.02236608  0.04834395 -0.03019028 -0.04912636  0.07963498
 -0.01571726  0.02258773  0.12637107  0.01002691  0.08798163 -0.09548923
  0.07042415  0.11434308  0.00902265  0.0682194  -0.04589128 -0.0315397
 -0.00427568  0.04286366 -0.06256895  0.05124556  0.08274208 -0.01155381
 -0.04605333 -0.07275854  0.05392071 -0.03980932 -0.07368367  0.04451564
  0.0362827  -0.04832048 -0.02009917 -0.08672954  0.10916894  0.10658942
  0.06261072  0.17693897  0.21654488  0.04464524  0.05964183  0.0178852
  0.01359412  0.06355514  0.01372807 -0.02966138  0.01300374 -0.01340398
 -0.07728048 -0.08147264  0.03711174 -0.04994174 -0.02077021  0.11506406
  0.13350497  0.01510224  0.0623071   0.0829863  -0.04087242  0.09213932
  0.02508393  0.09740812 -0.00694504 -0.04382166 -0.09073623  0.08050886
 -0.028761   -0.08823677 -0.0055057  -0.06118046  0.01278955 -0.0077752
 -0.00067417  0.04860631 -0.03766103 -0.0064845  -0.07

Save the result to a column

In [9]:
def get_review_vector(tokens):
    # Join tokens into a sentence (fastText expects a string, not list)
    sentence = " ".join(tokens)
    return model.get_sentence_vector(sentence)

# Apply to DataFrame
df["unweighted_vector"] = df["tokens"].apply(get_review_vector)

df

,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name,tokens,unweighted_vector
0,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses,"['high', 'hopes', 'wanted', 'work', 'initially...","[0.056675307, 0.030812785, 0.053851854, -0.004..."
1,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants,"['jumpsuit', 'fun', 'flirty', 'fabulous', 'tim...","[0.044263314, 0.013135654, 0.016781278, -0.009..."
2,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses,"['shirt', 'due', 'adjustable', 'front', 'tie',...","[0.053057756, 0.018640246, 0.04379628, -0.0091..."
3,1080,49,Not for the very petite,"I love tracy reese dresses, but this one is no...",2,0,4,General,Dresses,Dresses,"['tracy', 'reese', 'dresses', 'petite', 'feet'...","[0.060936436, 0.020342419, 0.04622086, 1.00809..."
4,858,39,Cagrcoal shimmer fun,I aded this in my basket at hte last mintue to...,5,1,1,General Petite,Tops,Knits,"['basket', 'hte', 'person', 'store', 'pick', '...","[0.049930412, 0.030952083, 0.050946318, -0.010..."
...,...,...,...,...,...,...,...,...,...,...,...,...
19657,1104,34,Great dress for many occasions,I was very happy to snag this dress at such a ...,5,1,0,General Petite,Dresses,Dresses,"['happy', 'snag', 'price', 'easy', 'slip', 'cu...","[0.04547508, 0.041090403, 0.031324998, 0.00296..."
19658,862,48,Wish it was made of cotton,"It reminds me of maternity clothes. soft, stre...",3,1,0,General Petite,Tops,Knits,"['reminds', 'maternity', 'clothes', 'stretchy'...","[0.056014594, 0.016797625, 0.041858178, 0.0026..."
19659,1104,31,"Cute, but see through","This fit well, but the top was very see throug...",3,0,1,General Petite,Dresses,Dresses,"['worked', 'glad', 'store', 'order', 'online']","[0.07674071, 0.028490331, 0.036898624, 0.00363..."
19660,1084,28,"Very cute dress, perfect for summer parties an...",I bought this dress for a wedding i have this ...,3,1,2,General,Dresses,Dresses,"['wedding', 'summer', 'medium', 'fits', 'waist...","[0.057828512, 0.016868837, 0.04380192, -0.0105..."


Save to a txt file with correct format

In [10]:
out_path = "unweighted_vectors.txt"

with open(out_path, "w", encoding="utf-8") as fout:
    for review_index, vec_str in enumerate(df["unweighted_vector"].values):
        # Convert string "[0.1, 0.2, ...]" into Python list
        fout.write(f"#{review_index}," + ",".join(map(str, vec_str)) + "\n")

print(f"[OK] Wrote {len(df)} lines to {out_path}")

# Preview first 3 lines
with open(out_path, "r", encoding="utf-8") as f:
    for _ in range(3):
        print(f.readline().rstrip())

[OK] Wrote 19662 lines to unweighted_vectors.txt
#0,0.056675307,0.030812785,0.053851854,-0.004569687,0.00679456,0.025780503,0.014843241,0.012028893,0.021787003,0.0052560396,0.011487953,0.03815299,-0.015979273,0.008874391,0.08518326,0.017684355,0.011057767,0.039421696,-0.018902756,0.016244316,-0.038014166,-0.025049832,-0.024890602,0.053263772,-0.045133576,0.01494127,-0.0035772587,0.0017018539,0.03601538,-0.04115634,0.008940749,0.07489644,0.00036660113,-0.01969987,0.027574763,0.003982384,-0.01237831,0.0034708753,0.00071420794,-0.002122232,0.03315569,-0.0312842,0.033304807,-0.026655057,-0.0043801176,0.062455777,-0.053581588,-0.05202014,0.040936355,-0.031181037,-0.020509318,-0.017710617,0.053659394,0.0041956627,0.021706201,0.023675703,0.016264433,0.008566323,-0.05266789,-0.09035791,0.043441933,0.035167135,0.0035077578,0.013684596,-0.0023495941,0.03126316,0.052147273,0.007216585,-0.055959392,0.029443573,-0.012245622,-0.025329292,-0.0008238809,0.00024332345,0.020977244,0.004200558,0.03662112

Reload the corpus for TF-IDF fit transformation

In [11]:
# load corpus into sentences for vectorizer
with open("corpus.txt", "r", encoding="utf-8") as f:
    sentences = [line.strip() for line in f]

# Create the TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(sentences)

Use TF-IDF weight to vectorize the tokens

In [12]:
# Map token indices to their corresponding tokens
index_to_token = {j: t for t, j in tfidf_vectorizer.vocabulary_.items()}

# Get the embedding dimension from the model
embedding_dimension = model.get_dimension()

# Create a weight matrix for the embeddings
weight_matrix = np.zeros((tfidf_matrix.shape[0], embedding_dimension))

for i in range(tfidf_matrix.shape[0]):
    row = tfidf_matrix.getrow(i)
    if row.nnz == 0:
        continue

    vector_sum = np.zeros(embedding_dimension, dtype=np.float32)
    weight_sum = 0.0

    # Compute the weighted sum of word vectors
    for j, w in zip(row.indices, row.data):
        token = index_to_token[j]
        vec = model.get_word_vector(token)
        vector_sum += vec * float(w)
        weight_sum += float(w)

    # Normalize the vector by the weight sum
    if weight_sum > 0:
        weight_matrix[i] = vector_sum / weight_sum



Save the result

In [13]:
df["weighted_vector"] = [vec.tolist() for vec in weight_matrix]
df.head()

,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name,tokens,unweighted_vector,weighted_vector
0,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses,"['high', 'hopes', 'wanted', 'work', 'initially...","[0.056675307, 0.030812785, 0.053851854, -0.004...","[-0.15194080770015717, 0.2709076702594757, -0...."
1,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants,"['jumpsuit', 'fun', 'flirty', 'fabulous', 'tim...","[0.044263314, 0.013135654, 0.016781278, -0.009...","[-0.19007307291030884, 0.2583615779876709, -0...."
2,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses,"['shirt', 'due', 'adjustable', 'front', 'tie',...","[0.053057756, 0.018640246, 0.04379628, -0.0091...","[-0.1005202978849411, 0.24748192727565765, -0...."
3,1080,49,Not for the very petite,"I love tracy reese dresses, but this one is no...",2,0,4,General,Dresses,Dresses,"['tracy', 'reese', 'dresses', 'petite', 'feet'...","[0.060936436, 0.020342419, 0.04622086, 1.00809...","[-0.10557624697685242, 0.23336651921272278, -0..."
4,858,39,Cagrcoal shimmer fun,I aded this in my basket at hte last mintue to...,5,1,1,General Petite,Tops,Knits,"['basket', 'hte', 'person', 'store', 'pick', '...","[0.049930412, 0.030952083, 0.050946318, -0.010...","[-0.11893583089113235, 0.13134421408176422, 0...."


In [14]:
out_path = "weighted_vectors.txt"

with open(out_path, "w", encoding="utf-8") as fout:
    for review_index, vec_str in enumerate(df["weighted_vector"].values):
        # Convert string "[0.1, 0.2, ...]" into Python list
        fout.write(f"#{review_index}," + ",".join(map(str, vec_str)) + "\n")

print(f"[OK] Wrote {len(df)} lines to {out_path}")

# Preview first 3 lines
with open(out_path, "r", encoding="utf-8") as f:
    for _ in range(3):
        print(f.readline().rstrip())

[OK] Wrote 19662 lines to weighted_vectors.txt
#0,-0.15194080770015717,0.2709076702594757,-0.16623243689537048,-0.060854893177747726,0.024433080106973648,0.014817753806710243,-0.07676483690738678,0.07763008773326874,0.10281585901975632,-0.08245109766721725,-0.06947685778141022,0.15743495523929596,-0.06013061851263046,0.06431859731674194,0.3038192093372345,-0.01096080057322979,0.2607077956199646,-0.2240564376115799,0.1280156373977661,0.3168621361255646,0.02713979221880436,0.16708476841449738,-0.09974221140146255,-0.07126704603433609,0.013501224108040333,0.12450582534074783,-0.14467363059520721,0.11944281309843063,0.1810077279806137,-0.033353131264448166,-0.10613322257995605,-0.14924748241901398,0.11254014819860458,-0.08296165615320206,-0.17267903685569763,0.10324160754680634,0.09418461471796036,-0.1231081560254097,-0.026634039357304573,-0.16603918373584747,0.27814486622810364,0.19566558301448822,0.13649868965148926,0.38408151268959045,0.4492829144001007,0.10507085174322128,0.10422141849

## Task 3. Clothing Review Classification

...... Sections and code blocks on buidling classification models based on different document feature represetations. 
Detailed comparsions and evaluations on different models to answer each question as per specification. 

<span style="color: red"> You might have complex notebook structure in this section, please feel free to create your own notebook structure. </span>

In [ ]:
# Code to perform the task...


## Summary
Give a short summary and anything you would like to talk about the assessment tasks here.

## Couple of notes for all code blocks in this notebook
- please provide proper comment on your code
- Please re-start and run all cells to make sure codes are runable and include your output in the submission.   
<span style="color: red"> This markdown block can be removed once the task is completed. </span>